functions used:

*NumPy*:
- np.array
- np.zeros
- np.ones
- np.random.rand

In [35]:
import numpy as np
import pandas as pd
from scipy.io  import loadmat

#### Matrix Algebra

In [36]:
# using numpy to create an array
x = np.array([[1, 2, 3], [4, 5, 6]])

# another option is to create matrices using the following functions:
zeros = np.zeros((2,3)) # matrix of zeros
ones = np.ones((2,3)) # matrix of ones
identity = np.eye(3) # identity. matrix
rand = np.random.rand(2,3) # random nums between 0 nd 1

x

array([[1, 2, 3],
       [4, 5, 6]])

In [37]:
3*x

array([[ 3,  6,  9],
       [12, 15, 18]])

In [38]:
3 * x/2

array([[1.5, 3. , 4.5],
       [6. , 7.5, 9. ]])

##### Matrix addition and subtraction

In [39]:
x = np.array([[1,2],[3,4],[5,6]])
y = np.array([[6,5],[4,3],[2,1]])

x, y

(array([[1, 2],
        [3, 4],
        [5, 6]]),
 array([[6, 5],
        [4, 3],
        [2, 1]]))

In [40]:
x + y

array([[7, 7],
       [7, 7],
       [7, 7]])

In [41]:
x - y

array([[-5, -3],
       [-1,  1],
       [ 3,  5]])

##### Matrix Multiplication

- `@` symbol is for matrix multiplication (or `np.dot`)
- `*` symbol is for element wise multiplication
- `.T` is the transpose of a matrix

In [42]:
# (3x2) @ (2x3) = (3x3)
x @ y.T


array([[16, 10,  4],
       [38, 24, 10],
       [60, 38, 16]])

In [43]:
# (2x3) @ (3x2) = (2x2)
x.T @ y

array([[28, 19],
       [40, 28]])

Also works for lists

In [44]:
df = pd.read_parquet('data/df_SIO_TEMP.parquet')
temps = df['T'].values[:150]
temps

array([11.5       , 11.5       , 11.9       , 11.5       , 11.7       ,
       11.9       , 12.03333333, 12.16666667, 12.3       , 12.2       ,
       12.2       , 12.1       , 12.2       , 11.9       , 11.6       ,
       11.3       , 11.3       , 11.7       , 11.7       , 11.5       ,
       11.55      , 11.6       , 12.3       , 11.7       , 11.9       ,
       11.7       , 11.9       , 11.9       , 11.9       , 11.9       ,
       11.9       , 11.9       , 11.9       , 12.        , 12.05      ,
       12.1       , 12.5       , 11.9       , 12.3       , 12.4       ,
       12.7       , 12.1       , 12.5       , 12.5       , 13.7       ,
       12.5       , 13.        , 12.83333333, 12.66666667, 12.5       ,
       12.5       , 12.9       , 13.15      , 13.4       , 13.9       ,
       13.9       , 13.9       , 13.7       , 13.1       , 13.9       ,
       13.5       , 13.5       , 13.7       , 13.9       , 13.3       ,
       13.5       , 13.9       , 14.1       , 13.85      , 13.6 

#### computing percentiles

In [45]:
# 99th percentile
p99 = np.percentile(temps, 99)
p01 = np.percentile(temps, 1)

print("The 99th percentile is:", p99, 'C')
print("The 1st percentile is:", p01, 'C')

The 99th percentile is: 17.3 C
The 1st percentile is: 11.398 C


#### Pandas GroupBy

In [46]:
# Let's find the average temperature (T) for every year in our dataset
yearly_avg = df.groupby('year')['T'].mean()

print("Average Summer Temp by Year:")
print(yearly_avg)

Average Summer Temp by Year:
year
1917    16.713973
1918    17.380274
1919    16.901507
1920    16.554508
1921    16.572329
          ...    
2021    17.852877
2022    18.289863
2023    17.596301
2024    17.871038
2025    17.971875
Name: T, Length: 109, dtype: float64


In [47]:
# Multiple Statistics at once
# Let's find the Hottest and Coldest day of every year
summer_stats = df.groupby('year')['T'].agg(['min', 'max'])

print("Annual Temperature Extremes:")
print(summer_stats)

Annual Temperature Extremes:
       min   max
year            
1917  11.3  23.1
1918  12.5  22.9
1919  11.7  22.8
1920  12.0  22.6
1921  12.5  23.3
...    ...   ...
2021  13.1  24.9
2022  13.5  24.8
2023  12.4  23.6
2024  12.9  24.6
2025  12.5  24.1

[109 rows x 2 columns]


In [48]:
# If you want to see how many days are recorded for each month across all years:
month_counts = df.groupby('month').size()

print("Number of days recorded per month:")
print(month_counts)

Number of days recorded per month:
month
1     3379
2     3079
3     3379
4     3270
5     3379
6     3270
7     3379
8     3379
9     3270
10    3379
11    3240
12    3348
dtype: int64


#### Form a 2-dimensional array of T summer day vs year using a for loop   


In [49]:
yrs = df['year'].unique() # .unique() gets unique values
nyr = len(yrs)
nday = 31 + 31 + 30 # July (31) + Aug (31) + Sep (30)

# create a matrix to hold the data 
Tsummer = np.zeros((nday, nyr))

for j, year in enumerate(yrs):
    # Filter for months 7, 8, and 9 for the specific year
    summer_data = df[(df.index.month >= 7) & 
                     (df.index.month <= 9) & 
                     (df.index.year == year)]['T']
    
    # Fill the column (ensure the lengths match)
    Tsummer[:, j] = summer_data.values[:nday]


#### Same process using pivot tables (perks of Pandas)

In [ ]:
# filter for summer
summer_df = df[(df['month'] >= 7) & (df['month'] <= 9)]

# assign the year and the day-count-within-summer
# cumcount() gives a running count within each group 

summer_df['yr'] = summer_df['year']
summer_df['day_idx'] = summer_df.groupby('yr').cumcount()

# pivot: Rows = day_idx, Columns = yr, Values = T
Tsummer_df = summer_df.pivot(index='day_idx', columns='yr', values='T')
Tsummer=Tsummer_df.to_numpy() # makes into an array
Tsummer

/var/folders/7q/04p1sn755vg5x39skt2hykjh0000gn/T/ipykernel_13917/4086787945.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summer_df['yr'] = summer_df['year']
/var/folders/7q/04p1sn755vg5x39skt2hykjh0000gn/T/ipykernel_13917/4086787945.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summer_df['day_idx'] = summer_df.groupby('yr').cumcount()


array([[21.1 , 20.1 , 20.2 , ..., 21.1 , 19.6 , 17.1 ],
       [21.3 , 19.8 , 20.8 , ..., 20.5 , 21.7 , 18.2 ],
       [22.1 , 20.2 , 20.8 , ..., 18.3 , 21.7 , 18.1 ],
       ...,
       [19.5 , 18.9 , 19.2 , ..., 19.  , 16.7 , 18.7 ],
       [18.7 , 19.2 , 19.6 , ..., 19.9 , 16.85, 20.1 ],
       [18.9 , 19.5 , 19.  , ..., 20.9 , 17.  , 19.7 ]])

In [52]:
Tsummer

array([[21.1 , 20.1 , 20.2 , ..., 21.1 , 19.6 , 17.1 ],
       [21.3 , 19.8 , 20.8 , ..., 20.5 , 21.7 , 18.2 ],
       [22.1 , 20.2 , 20.8 , ..., 18.3 , 21.7 , 18.1 ],
       ...,
       [19.5 , 18.9 , 19.2 , ..., 19.  , 16.7 , 18.7 ],
       [18.7 , 19.2 , 19.6 , ..., 19.9 , 16.85, 20.1 ],
       [18.9 , 19.5 , 19.  , ..., 20.9 , 17.  , 19.7 ]])